# Шаг 04 — инструменты: модель просит, код выполняет

Инструмент — обычная Python-функция. `@tool` превращает её в **JSON-схему**, которую видит модель.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from pathlib import Path

from langchain_core.tools import tool


@tool(parse_docstring=True)
def read_file(path: str) -> str:
    """Read a text file and return its content (first 200 lines).

    Args:
        path: File path relative to the current directory, e.g. "src/app.py".
    """
    file = Path(path)
    if file.name.startswith(".env"):
        return "Error: access to secrets is not allowed"
    if not file.is_file():
        return f"Error: {path} is not a file"
    return "\n".join(file.read_text(errors="replace").splitlines()[:200])

## Что видит модель

Модель никогда не видит Python-код — только эту схему. Имя функции → `name`, докстринг → `description`, type hints и раздел `Args:` → `parameters`.

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(read_file), indent=2, ensure_ascii=False))

Инструмент по-прежнему обычная функция — можно вызвать руками:

In [ ]:
print(read_file.invoke({"path": "pyproject.toml"})[:300])

## 1. Модель просит вызвать инструмент

`bind_tools` прикладывает схему к запросу. Модель ничего не запускает сама — она отвечает `tool_calls`.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

llm_with_tools = llm.bind_tools([read_file])

messages = [HumanMessage("Прочитай файл pyproject.toml и перечисли зависимости проекта.")]
ai = llm_with_tools.invoke(messages)
messages.append(ai)

print("text:", repr(ai.content))
ai.tool_calls

## 2. Наш код выполняет вызов и возвращает результат

In [ ]:
for call in ai.tool_calls:
    result = read_file.invoke(call["args"])
    messages.append(ToolMessage(result, tool_call_id=call["id"]))
    print(f"ran {call['name']}({call['args']}) → {len(result)} chars")

## 3. Модель читает результат и отвечает

In [ ]:
final = llm_with_tools.invoke(messages)
messages.append(final)
print(final.content or final.tool_calls)

Вся переписка: вопрос → запрос инструмента → результат → ответ. Мы сделали ровно **один** раунд — если модели нужно больше, нужен цикл (шаг 05).

In [ ]:
for message in messages:
    print(type(message).__name__)

## Под капотом: что улетает в API и что видит модель

```
HumanMessage + @tool → JSON-запрос (формат OpenAI) → OpenRouter → провайдер
  → шаблон модели превращает JSON в ТЕКСТ → модель читает и пишет текст
  → провайдер парсит текст обратно в JSON tool_calls → LangChain → AIMessage
```

Посмотрим на каждый слой по очереди.

### 1. JSON-запрос, который отправляет `.invoke()`

In [ ]:
request = llm._get_request_payload(messages[:1], **llm_with_tools.kwargs)
print(json.dumps(request, indent=2, ensure_ascii=False))

Второй запрос — после инструмента. В нём уже вызов модели (`assistant` + `tool_calls`) и наш результат (`role: tool`), связанные одним `id`:

In [ ]:
request2 = llm._get_request_payload(messages[:3], **llm_with_tools.kwargs)
print(json.dumps(request2["messages"], indent=2, ensure_ascii=False))

### 2. Сырой HTTP-ответ провайдера — до того, как LangChain его разберёт

Отправляем тот же JSON сами, обычным HTTP-запросом:

In [ ]:
import httpx

response = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    json=request,
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
    timeout=60,
)
raw = response.json()          # response.text — это буквально байты ответа
print("provider:", raw.get("provider"))
print(json.dumps(raw["choices"][0], indent=2, ensure_ascii=False))

Заметь: `arguments` — это **строка** с JSON внутри. LangChain делает `json.loads` и кладёт результат в `ai.tool_calls[0]["args"]`.

### 3. Текст, который на самом деле читает и пишет модель

Модель понимает только строку токенов. Сервер провайдера собирает её из `messages` и `tools` по **chat template** модели. Возьмём официальный шаблон Gemma 4 с HuggingFace (нужен интернет, скачивается только токенизатор) и отрендерим тот же запрос:

In [ ]:
os.environ["TRANSFORMERS_VERBOSITY"] = "error"   # no "PyTorch not found" noise — we only need the tokenizer
os.environ["HF_HUB_VERBOSITY"] = "error"
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # download progress bar needs ipywidgets
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-26b-a4b-it")   # HF repo of our model
tools_json = [convert_to_openai_tool(read_file)]

print(tokenizer.apply_chat_template(
    [{"role": "user", "content": messages[0].content}],
    tools=tools_json, add_generation_prompt=True, tokenize=False,
))

Схема инструмента превратилась в собственный формат Gemma (`<|tool>declaration:…`), а не JSON. А вот как выглядят **вызов инструмента**, который сгенерировала модель, и наш результат — тоже просто текст со спецтокенами:

In [ ]:
conversation = [
    {"role": "user", "content": messages[0].content},
    {"role": "assistant", "tool_calls": [
        {"type": "function", "function": {"name": c["name"], "arguments": c["args"]}}
        for c in ai.tool_calls
    ]},
    {"role": "tool", "name": ai.tool_calls[0]["name"], "content": result[:350]},
]
text = tokenizer.apply_chat_template(conversation, tools=tools_json, tokenize=False)
print(text[text.index("<|turn>model"):])

**Итог.**

- Модель **не видит JSON**: схема превращается в формат конкретной модели — у Gemma, Qwen, Llama он свой. OpenAI-JSON — лишь общий «язык» API поверх них.
- «Вызов инструмента» — это обычный сгенерированный текст `<|tool_call>call:read_file{…}<tool_call|>`; сервер провайдера находит его и превращает в `tool_calls`.
- Сам генерируемый текст через OpenRouter не увидеть — провайдер отдаёт уже JSON. Выше — точная реконструкция тем же шаблоном; вживую — только на своей модели (vLLM, Ollama).